# Individual Assignment 1 — Computer Vision (H02A5a)
### Air Hockey: From Basic Image Processing to Object Detection

**Video content:** Two people playing air hockey. The orange puck is the primary tracked object; blue and red strikers are also detected in later sections.

**Structure:**
| Time | Section |
|------|---------|
| 0–4s | Colour vs. Grayscale |
| 4–12s | Smoothing: Gaussian vs. Bilateral |
| 12–20s | Foreground Extraction (RGB → HSV → Morphology) |
| 20–25s | Sobel Edge Detection |
| 25–35s | Hough Circle Transform |
| 35–40s | Template Matching |
| 40–60s | Carte Blanche (Color shift, Motion Trail, Collision Flash) |

In [2]:
import argparse
import cv2
import sys
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

We define a couple of helper functions to check whether a frame is between two time points (in milliseconds).

In [3]:
def get_frame_time(cap) -> int:
  return int(cap.get(cv2.CAP_PROP_POS_MSEC))

def between(cap, lower: int, upper: int) -> bool:
    return lower <= get_frame_time(cap) < upper

## Section 2.1 — Basic Image Processing Functions
These functions implement the effects used in the first 20 seconds of the video.

- `grayscale` — converts BGR → Gray → BGR (keeps 3 channels for the writer)
- `gaussian_blur` / `bilateral_blur` — smoothing filters with configurable kernel size
- `median_blur` — defined for experimentation; not used in the final video
- `horizontal_edges` / `vertical_edges` / `edges` — Sobel-based coloured edge maps
- `edges_on_masked_frame` — Sobel applied only within the orange HSV mask (puck-only edges)

In [4]:
def grayscale(frame):
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_frame = cv2.cvtColor(gray_frame, cv2.COLOR_GRAY2BGR)
    return gray_frame

In [5]:
def gaussian_blur(frame, kernel):
    blurred_frame = cv2.GaussianBlur(frame, kernel, 0)
    return blurred_frame

In [6]:
def bilateral_blur(frame, kernel_size=5, sigma_color=75, sigma_space=75):
    blurred_frame = cv2.bilateralFilter(frame, kernel_size, sigma_color, sigma_space)
    return blurred_frame

In [7]:
def horizontal_edges(frame, threshold=50, kernel_size=3):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Apply Sobel in the Y-direction
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, dx=0, dy=1, ksize=kernel_size)  # cv2.CV_64F allows the math to hold negative numbers during calculation
    abs_sobel_y = cv2.convertScaleAbs(sobel_y)
    # Threshold the result to keep only the STRONG edges
    _, edge_mask = cv2.threshold(abs_sobel_y, threshold, 255, cv2.THRESH_BINARY)
    # Color the lines!
    horizontal_colored_edges = np.zeros_like(frame)
    horizontal_colored_edges[edge_mask == 255] = [0, 0, 255]
    return horizontal_colored_edges

In [8]:
def vertical_edges(frame, threshold=50, kernel_size=3):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Apply Sobel in the Y-direction
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, dx=1, dy=0, ksize=kernel_size) # cv2.CV_64F allows the math to hold negative numbers during calculation
    abs_sobel_x = cv2.convertScaleAbs(sobel_x)
    # Threshold the result to keep only the STRONG edges
    _, edge_mask = cv2.threshold(abs_sobel_x, threshold, 255, cv2.THRESH_BINARY)
    # Color the lines!
    vertical_colored_edges = np.zeros_like(frame)
    vertical_colored_edges[edge_mask == 255] = [255, 0, 0]
    return vertical_colored_edges

In [9]:
def edges(frame, threshold=50, kernel_size=3):
    vertical = vertical_edges(frame, threshold, kernel_size)
    horizontal = horizontal_edges(frame, threshold, kernel_size)
    combined = cv2.addWeighted(vertical, 1, horizontal, 1, 0)
    return combined

In [10]:
def edges_on_masked_frame(frame, threshold=50, kernel_size=3):
    """
    Sobel applied only to the orange-masked region.
    Zeroes out all non-orange pixels before running Sobel, so only
    edges belonging to the puck survive.
    Blue = vertical puck edges, Red = horizontal puck edges.

    Development notes:
    - v1: ran edges() directly on the full color frame --> every edge in the
      scene was detected (table frame, people, background furniture).
      Impossible to isolate puck edges from the noise.
    - v2: tried raising threshold to 150 on full frame --> reduced noise but
      background edges still dominated; puck edge indistinguishable.
    - v3 (current): apply HSV orange mask first, zero non-orange pixels,
      THEN run Sobel --> only orange-region edges survive. Dramatic reduction
      in false detections. This motivates using color preprocessing before
      any gradient-based method.
    """
    # Build orange mask (same parameters as detect_puck)
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([7, 100, 70]), np.array([20, 255, 255]))
    # Apply ROI: table surface only
    roi = np.zeros(frame.shape[:2], dtype=np.uint8)
    roi[130:330, 90:760] = 255
    mask = cv2.bitwise_and(mask, roi)
    # Zero-out non-orange pixels so Sobel only sees the puck region
    masked_frame = cv2.bitwise_and(frame, frame, mask=mask)
    vertical   = vertical_edges(masked_frame,   threshold, kernel_size)
    horizontal = horizontal_edges(masked_frame, threshold, kernel_size)
    return cv2.addWeighted(vertical, 1, horizontal, 1, 0)

## Section 2.2 — Object Detection Functions

### Hough Circle Transform
`hough_transform_circles` runs Hough on raw grayscale — used to demonstrate the effect of parameters.  
`hough_transform_circles_puck` first applies an HSV orange mask, then runs Hough on the resulting edges — the key insight of this section.

### Puck & Striker Detection (used from 33s onward)
`detect_puck`, `detect_blue_striker`, `detect_red_striker` each:
1. Build a colour-specific HSV mask with a tight ROI
2. Apply morphological cleanup
3. Run Canny then HoughCircles on the clean mask
4. Return `(cx, cy, r)` or `None`

In [11]:
def hough_transform_circles(frame, dp = 1, minDist=200, param1=30, param2=8, minRadius=5, maxRadius=50):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_blurred = cv2.medianBlur(gray, 3)
    # Apply the Hough Circle Transform
    # - dp=1: Resolution of the accumulator (keep it at 1)
    # - minDist=200: Minimum distance between detected circle centers 
    # - param1=50: The upper threshold for the internal Canny edge detector
    # - param2=30: Lower means it finds more false circles.
    # - minRadius/maxRadius: Extremely helpful for filtering out garbage.
    circles = cv2.HoughCircles(
        gray_blurred, 
        cv2.HOUGH_GRADIENT, 
        dp=dp, 
        minDist=minDist, 
        param1=param1, 
        param2=param2, 
        minRadius=minRadius,
        maxRadius=maxRadius
    )

    # Draw the circles on a copy of the original image
    result = frame.copy()
    # Ensure at least some circles were found
    if circles is not None:
        # Convert the (x, y, r) coordinates and radius to integers
        circles = np.uint16(np.around(circles))
        for i in circles[0, :]:
            # Draw the outer circle edge
            cv2.circle(result, (i[0], i[1]), i[2], (0, 255, 0), 3)
            # Draw a tiny dot in the center of the circle
            cv2.circle(result, (i[0], i[1]), 2, (0, 0, 255), 3)

    return result

In [12]:
def hough_transform_circles_puck(frame, param1=30, param2=8, minRadius=8, maxRadius=25):
    """
    Development notes:
    - v1: ran HoughCircles on raw grayscale (hough_transform_circles).
      param2=25 --> detected 20+ circles all over the scene, including
      background furniture, hands, table frame.
    - v2: raised param2=50 --> fewer circles but still many false positives;
      parameter tuning alone proved insufficient on raw grayscale.
    - v3: added size constraints minRadius=8, maxRadius=20 --> eliminated large
      false positives but still missed the puck when scene was complex.
    - v4 (current): apply HSV orange color mask first, run Canny on the mask,
      feed those edges to HoughCircles --> near-zero false positives because
      Hough only ever sees orange-region edges. Key insight: feature
      preprocessing outperforms parameter tuning alone.
    """
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # Isolate orange puck
    lower_orange = np.array([5, 140, 140])
    upper_orange = np.array([22, 255, 255])
    mask = cv2.inRange(hsv, lower_orange, upper_orange)
    # Morphological cleanup to remove noise
    k = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    # Run Hough only on the edges of the color mask
    edges = cv2.Canny(mask, 50, 150)
    circles = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
        param1=param1, param2=param2, minRadius=minRadius, maxRadius=maxRadius)
    result = frame.copy()
    if circles is not None:
        circles = np.uint16(np.around(circles))
        # Keep only the largest/most confident circle (the puck)
        best = sorted(circles[0], key=lambda c: c[2], reverse=True)[0:10]
        for i in best:
            cv2.circle(result, (i[0], i[1]), i[2], (0, 255, 0), 3)
            cv2.circle(result, (i[0], i[1]), 2,       (0, 0, 255), 3)
    return result

In [13]:
def detect_puck(frame):
    """
    Development notes:
    - v1: ran HoughCircles on raw grayscale --> detected table frame, hands,
      background objects. 0% useful detections.
    - v2: applied HSV orange mask (H=10-22, S>120, V>80) with broad ROI
      [130:330, 90:760] --> good detection on table surface but x<160 region
      picked up the orange sweater sleeve near the left border, causing
      the collision flash to fire at the wrong location during carte blanche.
    - v3 (current): tightened ROI to [140:310, 160:740] --> eliminates the
      sweater false positive completely (tested: 0 false positives over 40-60s).
      Tradeoff: puck near the extreme edges of the table may be missed, but
      this is acceptable since the puck spends most time in the inner surface.
    - param2=6 (low): intentional -- the orange mask is already very selective,
      so a low accumulator threshold is safe and improves detection rate.
    - smallest circle strategy: among all candidates, pick min radius. The
      puck is always smaller than border decorations that leak through.
    """
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    h, w = frame.shape[:2]
    k3 = np.ones((3, 3), np.uint8)
    # Tighter ROI: excludes left border where orange sweater appears (x<160)
    roi_o = np.zeros((h, w), dtype=np.uint8)
    roi_o[140:310, 160:740] = 255
    mask_o = cv2.inRange(hsv, np.array([10, 120, 80]), np.array([22, 255, 255]))
    mask_o = cv2.bitwise_and(mask_o, roi_o)
    mask_o = cv2.morphologyEx(mask_o, cv2.MORPH_OPEN, k3)
    mask_o = cv2.morphologyEx(mask_o, cv2.MORPH_CLOSE, k3)
    edges_o = cv2.Canny(mask_o, 50, 150)
    c_o = cv2.HoughCircles(edges_o, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                        param1=30, param2=6, minRadius=6, maxRadius=20)
    if c_o is not None:
        best = min(c_o[0], key=lambda c: c[2])
        if best[2] <= 20:
            return tuple(best.astype(int))
    return None

In [14]:
def detect_blue_striker(frame):
    """
    Development notes:
    - v1: broad HSV blue range (H=90-130) with no ROI --> detected player jeans,
      background blue objects, and the far wall. Hough fired everywhere.
    - v2: narrowed hue to H=100-130, added saturation floor S>100 --> jeans
      partially excluded (low saturation denim) but background still leaked.
    - v3: added ROI [135:320, 80:760] to restrict to table surface --> jeans
      excluded because they appear above y=135. Background wall excluded too.
    - v4 (current): lowered V floor to 30 to handle striker in shadow, raised
      S floor to 100, used k5 morphology to join fragmented striker blob.
      Added y<305 guard to exclude the bottom border band where blue text appears.
    """
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    h, w = frame.shape[:2]
    k3 = np.ones((3, 3), np.uint8)
    k5 = np.ones((5, 5), np.uint8)
    roi_b = np.zeros((h, w), dtype=np.uint8)
    roi_b[135:320, 80:760] = 255
    mask_b = cv2.inRange(hsv, np.array([100, 100, 30]), np.array([130, 255, 220]))
    mask_b = cv2.bitwise_and(mask_b, roi_b)
    mask_b = cv2.morphologyEx(mask_b, cv2.MORPH_OPEN, k5)
    mask_b = cv2.morphologyEx(mask_b, cv2.MORPH_CLOSE, k5)
    edges_b = cv2.Canny(mask_b, 50, 150)
    c_b = cv2.HoughCircles(edges_b, cv2.HOUGH_GRADIENT, dp=1, minDist=30,
                           param1=30, param2=8, minRadius=10, maxRadius=28)
    blue = None
    if c_b is not None:
        for c in c_b[0]:
            if c[1] < 305:  # exclude bottom text/border band
                blue = tuple(c.astype(int))
                break
    return blue

In [15]:
def detect_red_striker(frame):
    """
    Development notes:
    - v1: single HSV red range (H=0-10) --> missed half the detections because
      red wraps around H=180 in OpenCV HSV. Fixed by adding a second range
      (H=172-180) and OR-ing the two masks.
    - v2: two-range mask applied but the table border frame is also red/dark-red.
      Hough detected the corner brackets of the frame as circles constantly.
    - v3: tried tightening S>180 --> rejected too many valid striker pixels
      (striker is partially in shadow), detection rate dropped to ~20%.
    - v4: added ROI [140:310, 430:760] restricting to right half of table only
      (striker stays in right half in this video). Helped but left-side border
      still bled through due to ROI overlap near center.
    - v5 (current): added circularity pre-filter before Hough. Computes
      contours on the red mask, keeps only blobs with circularity>0.35 and
      fill>0.30 and r<28. This removes the elongated table-edge blobs entirely
      since they have very low circularity (~0.1). Only round blobs reach Hough.
    """
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    h, w = frame.shape[:2]
    k5 = np.ones((5, 5), np.uint8)
    # --- RED STRIKER ---
    # Split ROI: left half + right half, skipping center post (x=330-490)
    roi_r = np.zeros((h, w), dtype=np.uint8)
    roi_r[140:310, 430:760] = 255
    m1 = cv2.inRange(hsv, np.array([0,   150, 60]), np.array([8,   255, 255]))
    m2 = cv2.inRange(hsv, np.array([172, 150, 60]), np.array([180, 255, 255]))
    mask_r = cv2.bitwise_and(cv2.bitwise_or(m1, m2), roi_r)
    mask_r = cv2.morphologyEx(mask_r, cv2.MORPH_OPEN, k5)
    mask_r = cv2.morphologyEx(mask_r, cv2.MORPH_CLOSE, k5)
    # Keep only circular contours before running Hough
    # This removes elongated table-edge blobs so Hough only sees round shapes
    circular_mask = np.zeros((h, w), dtype=np.uint8)
    contours, _ = cv2.findContours(mask_r, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        area = cv2.contourArea(c)
        if area < 40 or area > 800: 
            continue
        perim = cv2.arcLength(c, True)
        if perim == 0: 
            continue
        circularity = 4 * np.pi * area / (perim ** 2)
        (x, y), r = cv2.minEnclosingCircle(c)
        fill = area / (np.pi * r ** 2)
        if circularity > 0.35 and fill > 0.30 and r < 28:
            cv2.drawContours(circular_mask, [c], -1, 255, -1)  # paint it into clean mask
    # Now run Hough only on the pre-filtered circular blobs
    edges_r = cv2.Canny(circular_mask, 50, 150)
    c_r = cv2.HoughCircles(edges_r, cv2.HOUGH_GRADIENT, dp=1, minDist=30,
                        param1=30, param2=6, minRadius=8, maxRadius=28)
    red = tuple(c_r[0][0].astype(int)) if c_r is not None else None
    return red

## Section 2.2.3 — Template Matching

`orange_feature(img)` computes a feature map: the HSV saturation channel weighted by an orange hue mask. This gives a compact representation that is distinctive for the puck.

`get_puck_template(video_path)` crops that feature map around the puck's known location at t=5s to produce the template.

`template_match_puck(frame, template)` slides the template over the ROI using `cv2.TM_CCOEFF_NORMED` and returns the best-match centre plus a normalised likelihood heatmap.

In [16]:
def orange_feature(img):
    """Feature map: saturation weighted by orange hue mask."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([10, 80, 80]), np.array([22, 255, 255]))
    s = hsv[:,:,1].astype(np.float32)
    return s * (mask.astype(np.float32) / 255.0)

In [17]:
def get_puck_template(video_path):
    """Extract orange-feature template from a known good frame."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_MSEC, 5000)
    ret, frame = cap.read()
    cap.release()
    # Puck is at (670, 298) r=14 at t=5s
    cx, cy, r, pad = 670, 298, 14, 4
    feat = orange_feature(frame)
    return feat[cy-r-pad:cy+r+pad, cx-r-pad:cx+r+pad].copy()

In [18]:
def orange_mask(img):
    """Binary mask of orange pixels -->  shared by both Hough and template matching.
    
    Development notes:
    - v1: BGR threshold (R>140, G=50-170, B<130) --> also triggered on the
      red/orange table frame pole and wooden floor reflections. Too noisy for
      reliable puck isolation.
    - v2: switched to HSV H=7-20 -- hue channel separates orange from red
      (H<7) and yellow (H>20) regardless of lighting. Much cleaner.
    - v3: saturation floor S>120 added --> excludes pale/washed-out orange
      reflections on the metallic table surface.
    - v4 (current): ROI [130:330, 90:760] restricts mask to the inner table
      playing area, excluding the outer border decorations and background.
      MORPH_OPEN (removes small noise dots) followed by MORPH_CLOSE (fills
      small holes in the puck blob) gives a stable, clean mask.
      
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    roi = np.zeros(img.shape[:2], dtype=np.uint8)
    roi[130:330, 90:760] = 255
    mask = cv2.inRange(hsv, np.array([10, 120, 80]), np.array([22, 255, 255]))
    mask = cv2.bitwise_and(mask, roi)
    k3 = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k3)
    return mask

In [19]:
def template_match_puck(frame, template):
    """
    Returns (cx, cy, likelihood_map_uint8).
    Matching is done only within the table ROI to avoid border false positives.
    """
    h, w = frame.shape[:2]
    th, tw = template.shape[:2]
    roi_y1, roi_y2, roi_x1, roi_x2 = 120, 350, 80, 780
    feat = orange_feature(frame)
    roi_feat = feat[roi_y1:roi_y2, roi_x1:roi_x2]
    result = cv2.matchTemplate(roi_feat, template, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(result)
    # Best match center in full-frame coordinates
    cx = max_loc[0] + roi_x1 + tw // 2
    cy = max_loc[1] + roi_y1 + th // 2
    # Build full-frame likelihood map
    rh, rw = result.shape
    full_map = np.zeros((h, w), dtype=np.float32)
    full_map[roi_y1:roi_y1+rh, roi_x1:roi_x1+rw] = np.clip(result, 0, 1)
    # Normalize and smooth
    if full_map.max() > 0:
        vis = (full_map / full_map.max() * 255).astype(np.uint8)
    else:
        vis = np.zeros((h, w), dtype=np.uint8)
    vis = cv2.GaussianBlur(vis, (15, 15), 0)
    return cx, cy, vis

## Section 2.3 — Carte Blanche Effect Functions

- `change_puck_color` — shifts the hue of orange puck pixels in HSV space, cycling through the full spectrum
- `draw_trail` — renders the last N puck positions as a fading rainbow trail (green = oldest, red = newest)
- `collision_flash` — white overlay + 12-ray starburst at the puck location, fading over `FLASH_DURATION` frames

In [20]:
def change_puck_color(frame, puck, hue_shift):
    """Recolor puck pixels by shifting hue in HSV."""
    if puck is None:
        return frame
    cx, cy, r = puck
    result = frame.copy()
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # Circular mask restricted to orange pixels only
    puck_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.circle(puck_mask, (cx, cy), r + 3, 255, -1)
    puck_mask = cv2.bitwise_and(puck_mask, orange_mask(frame))
    # Shift hue and recolor only masked pixels
    hsv_shifted = hsv.copy()
    hsv_shifted[:,:,0] = (hsv_shifted[:,:,0].astype(int) + hue_shift) % 180
    recolored = cv2.cvtColor(hsv_shifted, cv2.COLOR_HSV2BGR)
    result[puck_mask > 0] = recolored[puck_mask > 0]
    return result

In [21]:
def draw_trail(frame, trail, puck_r):
    """Draw fading rainbow trail of past puck positions."""
    result = frame.copy()
    n = len(trail)
    for i, (tx, ty) in enumerate(trail):
        alpha = (i + 1) / n                     # 0=oldest, 1=newest
        radius = max(2, int(puck_r * alpha))
        hue = int((i / n) * 120)               # green (old) -> yellow -> red (new)
        color_hsv = np.uint8([[[hue, 255, 255]]])
        color_bgr = cv2.cvtColor(color_hsv, cv2.COLOR_HSV2BGR)[0][0]
        bgr = (int(color_bgr[0]), int(color_bgr[1]), int(color_bgr[2]))
        overlay = result.copy()
        cv2.circle(overlay, (tx, ty), radius, bgr, -1)
        cv2.addWeighted(overlay, alpha * 0.8, result, 1 - alpha * 0.8, 0, result)
    return result

In [22]:
def collision_flash(frame, puck, flash_intensity):
    """White flash overlay + starburst at puck location, fading with intensity 0→1."""
    result = frame.copy()
    overlay = np.ones_like(frame, dtype=np.uint8) * 255
    cv2.addWeighted(overlay, flash_intensity * 0.5, result, 1 - flash_intensity * 0.5, 0, result)
    if puck is not None:
        cx, cy = puck[0], puck[1]
        ray_len = int(50 * flash_intensity)
        for i in range(12):
            angle = (2 * np.pi * i) / 12
            x2 = int(cx + ray_len * np.cos(angle))
            y2 = int(cy + ray_len * np.sin(angle))
            cv2.line(result, (cx, cy), (x2, y2), (0, 255, 255), max(1, int(3 * flash_intensity)))
        cv2.circle(result, (cx, cy), int(20 * flash_intensity), (0, 255, 255), -1)
    return result

## Configuration

In [23]:
INPUT_FILE_NAME = 'video.mp4'
OUTPUT_FILE_NAME = 'konst_processed_video.mp4'
PUCK_TEMPLATE = get_puck_template(INPUT_FILE_NAME)
SHOW_FRAME_AT = -1   # Set this number to visualise the corresponding frame

## Main Processing Loop

`main()` reads the input video frame-by-frame, dispatches each frame to the appropriate processing block based on its timestamp, overlays subtitles, and writes the result to the output file.

The subtitle system uses three lines:
- **Line 1 (top, cyan):** section name and technique
- **Line 2 (bottom, white):** explanation of what the technique does
- **Line 3 (bottom, white, smaller):** current parameter values or observations

In [24]:
def main(input_video_file: str, output_video_file: str) -> None:
    # OpenCV video objects to work with
    cap = cv2.VideoCapture(input_video_file)

    if cap is None or not cap.isOpened():
        raise RuntimeError('The file was not found or is not a proper video.')

    fps = int(round(cap.get(cv2.CAP_PROP_FPS)))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_file, fourcc, fps, (frame_width, frame_height))

    puck_trail         = []
    MAX_TRAIL          = 25
    flash_frames_left  = 0
    FLASH_DURATION     = 8
    cooldown_left      = 0
    COLLISION_COOLDOWN = 15

    kernel = 5
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    with tqdm(total=total_frames, desc="Rendering Video", unit="frame") as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            subtitle_line1 = ""
            subtitle_line2 = ""
            subtitle_line3 = ""

            if ret:
                current_ms = get_frame_time(cap)
                original_frame = frame.copy()

                lower_bgr = np.array([0,  80, 170])
                upper_bgr = np.array([80, 190, 255])
                lower_hsv = np.array([7,  120,  80])
                upper_hsv = np.array([22, 255, 255])

                # ── 0-4s: Color vs Grayscale ─────────────────────────────────────
                if 0 <= current_ms < 1000:
                    subtitle_line1 = "Sec.1: Color vs Grayscale"
                    subtitle_line2 = "Original color frame -->  all 3 RGB channels active"

                elif 1000 <= current_ms < 2000:
                    frame = grayscale(frame)
                    subtitle_line1 = "Sec.1: Color vs Grayscale"
                    subtitle_line2 = "Grayscale: single luminance channel -->  color information lost"

                elif 2000 <= current_ms < 3000:
                    subtitle_line1 = "Sec.1: Color vs Grayscale"
                    subtitle_line2 = "Back to color -->  notice how hue helps distinguish objects"

                elif 3000 <= current_ms < 4000:
                    frame = grayscale(frame)
                    subtitle_line1 = "Sec.1: Color vs Grayscale"
                    subtitle_line2 = "Grayscale again -->  intensity structure preserved, hue gone"

                # ── 4-12s: Smoothing & Blurring ──────────────────────────────────
                elif 4000 <= current_ms < 5000:
                    subtitle_line1 = "Sec.2: Smoothing & Blurring"
                    subtitle_line2 = "Original frame -->  no filter applied, full texture visible"

                elif 5000 <= current_ms < 6000:
                    frame = gaussian_blur(frame, (kernel, kernel))
                    subtitle_line1 = "Sec.2: Gaussian Blur"
                    subtitle_line2 = "Averages pixels with a bell-shaped kernel -->  uniform softening"
                    subtitle_line3 = f"Kernel: {kernel}x{kernel}"

                elif 6000 <= current_ms < 7000:
                    frame = gaussian_blur(frame, (kernel + 2, kernel + 2))
                    subtitle_line1 = "Sec.2: Gaussian Blur"
                    subtitle_line2 = "Larger kernel = wider average = stronger blur, more detail lost"
                    subtitle_line3 = f"Kernel: {kernel+2}x{kernel+2}"

                elif 7000 <= current_ms < 8000:
                    subtitle_line1 = "Sec.2: Smoothing & Blurring"
                    subtitle_line2 = "No filter -->  compare texture sharpness with blurred frames"

                elif 8000 <= current_ms < 9000:
                    frame = bilateral_blur(frame, kernel_size=kernel + 4, sigma_color=75, sigma_space=75)
                    subtitle_line1 = "Sec.2: Bilateral Filter"
                    subtitle_line2 = "Smooths flat regions but preserves sharp edges -->  unlike Gaussian"
                    subtitle_line3 = f"Kernel: {kernel+4}, sigma_color=75, sigma_space=75"

                elif 9000 <= current_ms < 10000:
                    frame = bilateral_blur(frame, kernel_size=kernel + 6, sigma_color=75, sigma_space=75)
                    subtitle_line1 = "Sec.2: Bilateral Filter"
                    subtitle_line2 = "Wider kernel: smoother regions, table edges remain crisp"
                    subtitle_line3 = f"Kernel: {kernel+6}, sigma_color=75, sigma_space=75"

                elif 10000 <= current_ms < 11000:
                    frame = gaussian_blur(frame, (kernel + 8, kernel + 8))
                    subtitle_line1 = "Sec.2: Gaussian Blur (large)"
                    subtitle_line2 = "Large Gaussian blurs edges too -->  bilateral at same kernel size preserves them"
                    subtitle_line3 = f"Kernel: {kernel+8}x{kernel+8}"

                elif 11000 <= current_ms < 12000:
                    frame = bilateral_blur(frame, kernel_size=kernel + 10, sigma_color=75, sigma_space=75)
                    subtitle_line1 = "Sec.2: Bilateral Filter (large)"
                    subtitle_line2 = "Max smoothing -->  noise gone, object boundaries still intact"
                    subtitle_line3 = f"Kernel: {kernel+10}, sigma_color=75, sigma_space=75"

                # ── 12-20s: Foreground Extraction ────────────────────────────────
                elif 12000 <= current_ms < 14000:
                    rgb_mask = cv2.inRange(frame, lower_bgr, upper_bgr)
                    frame = cv2.cvtColor(rgb_mask, cv2.COLOR_GRAY2BGR)
                    subtitle_line1 = "Sec.3: Foreground Extraction -->  RGB"
                    subtitle_line2 = "Threshold on R channel -->  noisy, sensitive to lighting changes"

                elif 14000 <= current_ms < 16000:
                    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
                    hsv_mask = cv2.inRange(hsv_frame, lower_hsv, upper_hsv)
                    frame = cv2.cvtColor(hsv_mask, cv2.COLOR_GRAY2BGR)
                    subtitle_line1 = "Sec.3: Foreground Extraction -->  HSV"
                    subtitle_line2 = "HSV separates hue from brightness -->  more robust to lighting"

                elif 16000 <= current_ms < 20000:
                    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
                    base_mask = cv2.inRange(hsv_frame, lower_hsv, upper_hsv)
                    morph_kernel = np.ones((9, 9), np.uint8)
                    improved_mask = cv2.morphologyEx(base_mask, cv2.MORPH_CLOSE, morph_kernel)
                    improvements = cv2.subtract(improved_mask, base_mask)
                    canvas = np.zeros_like(frame)
                    canvas[base_mask == 255] = [255, 255, 255]
                    canvas[improvements == 255] = [255, 0, 0]
                    frame = canvas
                    subtitle_line1 = "Sec.3: Morphological Improvement (MORPH_CLOSE)"
                    subtitle_line2 = "White = original mask | Blue = holes/gaps filled by closing"

                # ── 20-25s: Sobel Edge Detection ─────────────────────────────────
                elif 20000 <= current_ms < 25000:

                    # 20-21s: Full frame, low threshold -->  everything detected
                    if 20000 <= current_ms < 21000:
                        frame = edges(frame, threshold=50, kernel_size=3)
                        subtitle_line1 = "Sec.4: Sobel -->  Full Frame"
                        subtitle_line2 = "Low threshold: every edge in scene detected -->  table, people, background"
                        subtitle_line3 = "Threshold=50, Kernel=3 | Blue=vertical  Red=horizontal  Magenta=both"

                    # 21-22s: Full frame, higher threshold -->  less noise but still full scene
                    elif 21000 <= current_ms < 22000:
                        frame = edges(frame, threshold=100, kernel_size=3)
                        subtitle_line1 = "Sec.4: Sobel -->  Full Frame, Higher Threshold"
                        subtitle_line2 = "Raising threshold reduces noise -->  but background edges still dominate"
                        subtitle_line3 = "Threshold=100, Kernel=3 -->  puck edges indistinguishable from scene"

                    # 22-23s: Full frame, large kernel -->  floods the frame
                    elif 22000 <= current_ms < 23000:
                        frame = edges(frame, threshold=50, kernel_size=5)
                        subtitle_line1 = "Sec.4: Sobel -->  Full Frame, Large Kernel"
                        subtitle_line2 = "Kernel=5 computes gradient over wider area -->  edges flood and merge"
                        subtitle_line3 = "Threshold=50, Kernel=5 -->  over-detection, impossible to isolate puck"

                    # 23-24s: Orange-masked frame, low threshold -->  only puck region
                    elif 23000 <= current_ms < 24000:
                        frame = edges_on_masked_frame(frame, threshold=50, kernel_size=3)
                        subtitle_line1 = "Sec.4: Sobel -->  Orange Mask Applied First"
                        subtitle_line2 = "HSV mask zeroes non-orange pixels -->  Sobel now sees only puck region"
                        subtitle_line3 = "Threshold=50, Kernel=3 -->  background eliminated by preprocessing"

                    # 24-25s: Orange-masked frame, tuned threshold -->  clean puck edge
                    elif 24000 <= current_ms < 25000:
                        frame = edges_on_masked_frame(frame, threshold=80, kernel_size=3)
                        subtitle_line1 = "Sec.4: Sobel -->  Masked + Tuned Threshold"
                        subtitle_line2 = "Fine-tuned threshold on masked frame -->  light color edges appear only"
                        subtitle_line3 = "Threshold=80, Kernel=3 -->  this edge will feed the Hough transform next"

                # ── 25-35s: Hough Transform ──────────────────────────────────────
                elif 25000 <= current_ms < 35000:
                    if 25000 <= current_ms < 27000:
                        frame = hough_transform_circles(frame, dp=1, minDist=30,
                                                        param1=50, param2=50,
                                                        minRadius=5, maxRadius=150)
                        subtitle_line1 = "Sec.5: Raising Accumulator Threshold"
                        subtitle_line2 = "param2=50 still produces many false positives on raw grayscale"
                        subtitle_line3 = "param1=50  param2=50 -->  parameter tuning alone is insufficient"

                    # Step 3 (29-31s): Add size constraints -->  geometry as prior
                    # minRadius/maxRadius limits to puck-sized objects only
                    elif 27000 <= current_ms < 29000:
                        frame = hough_transform_circles(frame, dp=1, minDist=30,
                                                        param1=50, param2=50,
                                                        minRadius=10, maxRadius=25)
                        subtitle_line1 = "Sec.5: Adding Size Constraints"
                        subtitle_line2 = "Size + strict threshold on grayscale: nothing survives -->  motivates color masking"
                        subtitle_line3 = "minRadius=8, maxRadius=20, param2=50 -->  too strict without color preprocessing"

                    # Step 4 (31-33s): Color mask + Hough -->  the key insight
                    # Orange mask fed to Hough: dramatic improvement, only puck region survives
                    elif 29000 <= current_ms < 33000:
                        frame = hough_transform_circles_puck(frame, param1=30,
                                                             param2=10,
                                                             minRadius=6, maxRadius=20)
                        subtitle_line1 = "Sec.5: Color Mask + Hough"
                        subtitle_line2 = "Orange HSV mask preprocesses frame -->  background eliminated"
                        subtitle_line3 = "Feature preprocessing > parameter tuning alone"

                    # Step 5 (33-35s): Multi-object -->  three separate color masks
                    # Each object gets its own mask + Hough pass
                    elif 33000 <= current_ms < 35000:
                        puck = detect_puck(original_frame)
                        blue = detect_blue_striker(original_frame)
                        red  = detect_red_striker(original_frame)

                        subtitle_line1 = "Sec.5: Multi-Object Detection"
                        subtitle_line2 = "Three color masks (orange/blue/red) -->  each object detected independently"
                        subtitle_line3 = ""

                        if puck is not None:
                            cv2.circle(frame, (puck[0], puck[1]), puck[2], (0, 165, 255), 3)
                            cv2.circle(frame, (puck[0], puck[1]), 2, (0, 0, 255), 3)
                            subtitle_line3 += "Puck "
                        if blue is not None:
                            cv2.circle(frame, (blue[0], blue[1]), blue[2], (255, 0, 0), 3)
                            cv2.circle(frame, (blue[0], blue[1]), 2, (0, 0, 255), 3)
                            subtitle_line3 += "| Blue Striker "
                        if red is not None:
                            cv2.circle(frame, (red[0], red[1]), red[2], (0, 0, 255), 3)
                            cv2.circle(frame, (red[0], red[1]), 2, (0, 255, 255), 3)
                            subtitle_line3 += "| Red Striker"

                # ── 35-40s: Template Matching ────────────────────────────────────
                elif 35000 <= current_ms < 40000:
                    cx, cy, likelihood_map = template_match_puck(original_frame, PUCK_TEMPLATE)
                    th = PUCK_TEMPLATE.shape[0]

                    if 35000 <= current_ms < 37000:
                        half = th // 2 + 2
                        box_color = (0, 255, 0) if (current_ms // 200) % 2 == 0 else (0, 255, 255)
                        cv2.rectangle(frame, (cx-half, cy-half), (cx+half, cy+half), box_color, 3)
                        cv2.rectangle(frame, (cx-half-3, cy-half-3), (cx+half+3, cy+half+3), (255, 255, 255), 1)
                        subtitle_line1 = "Sec.6: Template Matching -->  Localisation"
                        subtitle_line2 = "Flashing box alternates green/cyan every 200ms -->  box = best match location"

                    elif 37000 <= current_ms < 40000:
                        frame = cv2.cvtColor(likelihood_map, cv2.COLOR_GRAY2BGR)
                        subtitle_line1 = "Sec.6: Template Matching -->  Likelihood Map"
                        subtitle_line2 = "Multiple bright spots = other orange regions in scene -->  brightest = puck"
                        subtitle_line3 = "Algorithm picks the global maximum as the predicted location"

                # ── 40-60s: Carte Blanche ────────────────────────────────────────
                elif 40000 <= current_ms < 60000:
                    puck = detect_puck(original_frame)
                    blue = detect_blue_striker(original_frame)
                    red = detect_red_striker(original_frame)

                    if puck is not None:
                        puck_trail.append((puck[0], puck[1]))
                        if len(puck_trail) > MAX_TRAIL:
                            puck_trail.pop(0)

                    if 40000 <= current_ms < 50000:
                        # First 10s: color change + trail
                        hue_shift = int(((current_ms - 40000) / 4000) * 180) % 180
                        frame = change_puck_color(frame, puck, hue_shift)
                        if len(puck_trail) > 1:
                            frame = draw_trail(frame, puck_trail, puck[2] if puck else 10)
                        subtitle_line1 = "Carte Blanche: Color Change + Motion Trail"
                        subtitle_line2 = "Puck hue cycles full spectrum every 4s via HSV shift"
                        subtitle_line3 = "Trail: green = oldest position | red = most recent"

                    elif 50000 <= current_ms < 55000:
                        # Middle 5s: trail only
                        if len(puck_trail) > 1:
                            frame = draw_trail(frame, puck_trail, puck[2] if puck else 10)
                        subtitle_line1 = "Carte Blanche: Motion Trail"
                        subtitle_line2 = "Last 25 puck positions drawn -->  shows trajectory and speed"
                        subtitle_line3 = "Note: puck near table edges may be missed -->  ROI excludes border to avoid clothing"

                    elif 55000 <= current_ms < 60000:
                        # Last 5s: trail + collision flash
                        if len(puck_trail) > 1:
                            frame = draw_trail(frame, puck_trail, puck[2] if puck else 10)

                        if cooldown_left == 0 and puck is not None and blue is not None:
                            d = np.sqrt((puck[0]-blue[0])**2 + (puck[1]-blue[1])**2)
                            if d < 70:
                                flash_frames_left = FLASH_DURATION
                                cooldown_left = COLLISION_COOLDOWN

                        if cooldown_left == 0 and puck is not None and red is not None:
                            d = np.sqrt((puck[0]-red[0])**2 + (puck[1]-red[1])**2)
                            if d < 70:
                                flash_frames_left = FLASH_DURATION
                                cooldown_left = COLLISION_COOLDOWN

                        if flash_frames_left > 0:
                            intensity = flash_frames_left / FLASH_DURATION
                            frame = collision_flash(frame, puck, intensity)
                            flash_frames_left -= 1

                        if cooldown_left > 0:
                            cooldown_left -= 1

                        subtitle_line1 = "Carte Blanche: Collision Flash"
                        subtitle_line2 = "Proximity trigger: puck within 70px of striker fires flash"
                        subtitle_line3 = "White overlay + yellow starburst fades over 8 frames"

                # ── Subtitle rendering ───────────────────────────────────────────
                if subtitle_line1 != "":
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    fh = frame_height

                    # TOP: section title -->  cyan, scale 0.75, top-left
                    cv2.putText(frame, subtitle_line1, (20, 30), font, 0.75,
                                (0, 0, 0), 4, cv2.LINE_AA)
                    cv2.putText(frame, subtitle_line1, (20, 30), font, 0.75,
                                (0, 220, 255), 2, cv2.LINE_AA)

                    # BOTTOM line 3: parameters / observations -->  white, scale 0.52
                    if subtitle_line3 != "":
                        cv2.putText(frame, subtitle_line3, (20, fh - 70), font, 0.52,
                                    (0, 0, 0), 3, cv2.LINE_AA)
                        cv2.putText(frame, subtitle_line3, (20, fh - 70), font, 0.52,
                                    (255, 255, 255), 1, cv2.LINE_AA)

                    # BOTTOM line 2: explanation -->  white, scale 0.52
                    if subtitle_line2 != "":
                        cv2.putText(frame, subtitle_line2, (20, fh - 40), font, 0.52,
                                    (0, 0, 0), 3, cv2.LINE_AA)
                        cv2.putText(frame, subtitle_line2, (20, fh - 40), font, 0.52,
                                    (255, 255, 255), 1, cv2.LINE_AA)

                # write frame that you processed to output
                out.write(frame)

                # (optional) display the resulting frame
                if get_frame_time(cap) == SHOW_FRAME_AT:
                    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                    plt.show()

                pbar.update(1)

            else:
                break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

In [25]:
main(INPUT_FILE_NAME, OUTPUT_FILE_NAME)

Rendering Video:   0%|          | 0/1818 [00:00<?, ?frame/s]

Rendering Video:   9%|▉         | 161/1818 [00:30<05:11,  5.32frame/s]


KeyboardInterrupt: 